# FINAL CLEAN-SPLIT TRAINING — ResNet152V2

**Run only after 01F has passed.**

This notebook trains **ResNet152V2 only** on:

`/content/drive/MyDrive/Cataract/Data_Clean_LeakageControlled_FINAL`

Fixed settings recovered from the original experiments:
- seed = 42
- 224×224 RGB
- rescale = 1/255
- batch size = 32
- train augmentation = horizontal + vertical flip
- Adam, learning rate 1e-4
- categorical cross-entropy
- 20 epochs
- class order = Cataract, Normal, Not Eye
- validation accuracy selects the best checkpoint
- original recovered layer-freezing cutoff for this model = 564

**Do not change these settings.**

In [ ]:
# CELL 1 — Setup, Drive, GPU, paths
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, time, json, random, platform
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, CSVLogger, TerminateOnNaN
from tensorflow.keras.preprocessing.image import ImageDataGenerator

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

MODEL_NAME = 'ResNet152V2'
FREEZE_FIRST = 564
EPOCHS = 20

PROJECT = Path('/content/drive/MyDrive/Cataract')
DATA = PROJECT / 'Data_Clean_LeakageControlled_FINAL'
TRAIN = DATA / 'Train'
VAL = DATA / 'Validation'
TEST = DATA / 'Test'

OUT = PROJECT / 'FINAL_REVISION_2026_08' / 'clean_split_models' / MODEL_NAME
OUT.mkdir(parents=True, exist_ok=True)

for p in [TRAIN, VAL, TEST]:
    assert p.exists(), f'STOP: Missing {p}'

gpus = tf.config.list_physical_devices('GPU')

print('Model:', MODEL_NAME)
print('TensorFlow:', tf.__version__)
print('Keras:', tf.keras.__version__ if hasattr(tf.keras,'__version__') else 'bundled')
print('GPU devices:', gpus)
print('Dataset:', DATA)
print('Output:', OUT)

if not gpus:
    raise RuntimeError(
        'STOP: No GPU detected. In Colab choose Runtime → Change runtime type → GPU, then rerun CELL 1.'
    )

env = {
    'model': MODEL_NAME,
    'seed': SEED,
    'epochs': EPOCHS,
    'tensorflow': tf.__version__,
    'keras': tf.keras.__version__ if hasattr(tf.keras,'__version__') else 'bundled',
    'python': platform.python_version(),
    'gpu_devices': [str(x) for x in gpus],
    'dataset': str(DATA),
    'freeze_first': FREEZE_FIRST,
}
with open(OUT / 'environment.json', 'w') as f:
    json.dump(env, f, indent=2)

print('\n✅ CELL 1 COMPLETE — GPU found and paths are correct.')

In [ ]:
# CELL 2 — Data generators and count check
CLASS_ORDER = ['Cataract', 'Normal', 'Not Eye']

def make_generators(seed=42):
    train_aug = ImageDataGenerator(
        rescale=1./255,
        horizontal_flip=True,
        vertical_flip=True
    )
    plain = ImageDataGenerator(rescale=1./255)

    train = train_aug.flow_from_directory(
        TRAIN,
        target_size=(224,224),
        color_mode='rgb',
        class_mode='categorical',
        classes=CLASS_ORDER,
        batch_size=32,
        shuffle=True,
        seed=seed,
        interpolation='nearest'
    )

    val = plain.flow_from_directory(
        VAL,
        target_size=(224,224),
        color_mode='rgb',
        class_mode='categorical',
        classes=CLASS_ORDER,
        batch_size=32,
        shuffle=False,
        interpolation='nearest'
    )

    test = plain.flow_from_directory(
        TEST,
        target_size=(224,224),
        color_mode='rgb',
        class_mode='categorical',
        classes=CLASS_ORDER,
        batch_size=32,
        shuffle=False,
        interpolation='nearest'
    )

    return train, val, test

train, val, test = make_generators(SEED)

print('\nClass indices:', train.class_indices)
print('Train images:', train.samples)
print('Validation images:', val.samples)
print('Test images:', test.samples)

assert train.samples == 8845, f'Unexpected Train count: {train.samples}'
assert val.samples == 2178, f'Unexpected Validation count: {val.samples}'
assert test.samples == 2587, f'Unexpected Test count: {test.samples}'

assert train.class_indices == {
    'Cataract': 0,
    'Normal': 1,
    'Not Eye': 2
}

print('\n✅ CELL 2 COMPLETE — final clean dataset counts are correct.')

In [ ]:
# CELL 3 — Build ResNet152V2 with recovered original head/freezing
def build_head(base, model_name):
    x = base.output

    if model_name == 'InceptionResNetV2':
        x = layers.Conv2D(32, (3,3), activation='relu', padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D((2,2), padding='same')(x)
        x = layers.Dropout(0.17)(x)
        x = layers.GlobalAveragePooling2D()(x)
    else:
        x = layers.Conv2D(32, (3,3), activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D((2,2))(x)
        x = layers.Dropout(0.17)(x)
        x = layers.Conv2D(64, (2,2), activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.30)(x)
    out = layers.Dense(3, activation='softmax', name='preds')(x)
    return Model(base.input, out)

apps = tf.keras.applications

APP_CLASS = {
    'MobileNetV2': apps.MobileNetV2,
    'DenseNet201': apps.DenseNet201,
    'InceptionResNetV2': apps.InceptionResNetV2,
    'ResNet152V2': apps.ResNet152V2,
    'Xception': apps.Xception,
}[MODEL_NAME]

tf.keras.backend.clear_session()
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

base = APP_CLASS(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

model = build_head(base, MODEL_NAME)

# Reproduce recovered original layer-freezing behavior.
for layer in model.layers:
    layer.trainable = True

for i in range(min(FREEZE_FIRST, len(model.layers))):
    model.layers[i].trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

trainability = pd.DataFrame([
    {
        'index': i,
        'layer': layer.name,
        'class': layer.__class__.__name__,
        'trainable': layer.trainable
    }
    for i, layer in enumerate(model.layers)
])

trainability.to_csv(
    OUT / 'layer_trainability.csv',
    index=False
)

print('Total model layers:', len(model.layers))
print('First trainable layer index:',
      next((i for i,l in enumerate(model.layers) if l.trainable), None))
print('Trainable parameters:', sum(np.prod(v.shape) for v in model.trainable_weights))
print('Non-trainable parameters:', sum(np.prod(v.shape) for v in model.non_trainable_weights))

print('\n✅ CELL 3 COMPLETE — model built and trainability evidence saved.')

In [ ]:
# CELL 4 — Train 20 epochs and save the best validation checkpoint
DONE = OUT / 'DONE.txt'

if DONE.exists():
    raise RuntimeError(
        f'STOP: This run is already marked complete: {DONE}. '
        'Do not retrain unless we intentionally decide to replace it.'
    )

BEST = OUT / 'best.keras'

callbacks = [
    ModelCheckpoint(
        BEST,
        monitor='val_accuracy',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    CSVLogger(OUT / 'history.csv'),
    TerminateOnNaN()
]

print('Starting training:', MODEL_NAME)
print('Epochs:', EPOCHS)
print('Best checkpoint:', BEST)

t0 = time.time()

history = model.fit(
    train,
    validation_data=val,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

training_seconds = time.time() - t0

(OUT / 'training_time_seconds.txt').write_text(
    str(training_seconds)
)

print('\nTraining time (minutes):', training_seconds / 60)
print('✅ CELL 4 COMPLETE — training finished and best checkpoint saved.')

In [ ]:
# CELL 5 — Save untouched Test predictions from BEST checkpoint
# The checkpoint was selected only by validation accuracy.
best_model = tf.keras.models.load_model(
    BEST,
    compile=False
)

test.reset()

probs = best_model.predict(
    test,
    verbose=1
)

y_true = test.classes.copy()
y_pred = probs.argmax(axis=1)

np.save(OUT / 'probs.npy', probs)
np.save(OUT / 'y_true.npy', y_true)
np.save(OUT / 'y_pred.npy', y_pred)

pd.DataFrame({
    'filepath': test.filepaths,
    'y_true': y_true,
    'y_pred': y_pred
}).to_csv(
    OUT / 'test_predictions_index.csv',
    index=False
)

DONE.write_text(
    'completed\n'
    f'model={MODEL_NAME}\n'
    f'seed={SEED}\n'
    f'epochs={EPOCHS}\n'
    f'training_seconds={training_seconds}\n'
)

print('Prediction array shape:', probs.shape)
print('y_true shape:', y_true.shape)
print('y_pred shape:', y_pred.shape)
print('Saved to:', OUT)
print('\n✅ DONE —', MODEL_NAME, 'final clean-split run is complete.')